# Paper 1 — Golden Age Semantic Reconfiguration
**Phase 5 — corrected reproducible rebuild**

Fixes:
- repeated identical chronology assignments are idempotent;
- contradictory assignments still raise an error;
- Góngora exact links are confidence **A**, fuzzy/variant links are **B**.

Composition, circulation, and sensitivity clocks remain separate. Do not build semantic networks yet.


In [ ]:
import sys,re,shutil,subprocess,unicodedata
from pathlib import Path
from difflib import SequenceMatcher
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES={
"navarro_tei":("https://github.com/bncolorado/CorpusSonetosSigloDeOro.git","092a5fe70a4065a4d84bfed288bffd3851348f9c"),
"gongora_scholarly":("https://github.com/gongoradigital/gongoraobra.git","3beadeecc059a7cc48499dc2683bb378a2630978"),
"herrera_stylistics":("https://github.com/lamusadecima/Digital-Stylistics-Applied-to-Golden-Age.git","0de990eac908897b5e931aeb5c496170ccf35bab"),
}
ROOT=Path("/content/gasr_phase5_sources"); ROOT.mkdir(exist_ok=True)
def clone(name,url,commit):
    dst=ROOT/name
    if dst.exists(): shutil.rmtree(dst)
    subprocess.run(["git","clone","--quiet",url,str(dst)],check=True)
    subprocess.run(["git","-C",str(dst),"checkout","--quiet",commit],check=True)
    got=subprocess.check_output(["git","-C",str(dst),"rev-parse","HEAD"],text=True).strip()
    assert got==commit
    return dst
paths={k:clone(k,*v) for k,v in SOURCES.items()}
N,G,HS=paths["navarro_tei"],paths["gongora_scholarly"],paths["herrera_stylistics"]
XML_ID="{http://www.w3.org/XML/1998/namespace}id"
def local(tag): return tag.split("}")[-1] if "}" in tag else tag
def el_text(el): return "" if el is None else " ".join(" ".join(el.itertext()).split())
def norm(s):
    s=unicodedata.normalize("NFKD",str(s))
    s="".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]","",s.lower())
def years(s): return sorted(set(int(x) for x in re.findall(r"(?<!\d)(1[56]\d{2})(?!\d)",str(s)) if 1580<=int(x)<=1626))
def blocks(path):
    raw=Path(path).read_text(encoding="utf-8",errors="replace").replace("\r\n","\n")
    out=[]
    for i,b in enumerate(re.split(r"\n\s*\n+",raw),1):
        ls=[x.strip() for x in b.splitlines() if x.strip()]
        if ls:
            txt="\n".join(ls)
            out.append({"block_id":i,"n_lines":len(ls),"text":txt,"signature":norm(txt),"first_line":ls[0],"first2_signature":norm("\n".join(ls[:2]))})
    return pd.DataFrame(out)
print("Pinned sources ready")


In [ ]:
def body_title(root):
    for body in root.iter():
        if local(body.tag)=="body":
            for x in body.iter():
                if local(x.tag)=="title":
                    t=el_text(x)
                    if t: return t
            break
    return ""
rows=[]
for fp in sorted(N.rglob("*.xml")):
    root=ET.parse(fp).getroot()
    ls=[el_text(x) for x in root.iter() if local(x.tag)=="l"]
    ls=[x for x in ls if x]
    if not ls: continue
    a=fp.parent.name; txt="\n".join(ls)
    bib=[" ".join(el_text(x).split()) for x in root.iter() if local(x.tag) in {"bibl","witness"} and el_text(x)]
    rows.append({"n_id":f"{a}::{fp.name}","author_dir":a,"title":body_title(root),"n_lines":len(ls),"text":txt,
                 "signature":norm(txt),"first_line":ls[0],"first_line_sig":norm(ls[0]),"first2_signature":norm("\n".join(ls[:2])),
                 "source_bibl":" | ".join(bib[:4]),"source_file":str(fp.relative_to(N))})
n=pd.DataFrame(rows); assert len(n)==5078
priority_A={"GarcilasoDeLaVega","JuanBoscan","FernandoDeHerrera","PedroEspinosa","JuanDeArguijo","JuanDeJauregui","LuisCarrilloySotomayor","Cervantes","Gongora","LopeDeVega_1","LopeDeVega_2","Quevedo"}
print("Navarro poems:",len(n),"| folders:",n.author_dir.nunique())

temporal=n[["n_id","author_dir","title","source_file","first_line"]].copy()
for c in ["composition_min","composition_max","circulation_year","sensitivity_min","sensitivity_max"]: temporal[c]=pd.NA
for c,v in [("temporal_confidence","unassigned"),("temporal_basis",""),("temporal_source",""),("chronology_status","undated"),
            ("circulation_basis",""),("circulation_source",""),("sensitivity_basis",""),("sensitivity_source",""),("sensitivity_status","none")]:
    temporal[c]=v

def assign_primary(ids,lo,hi,conf,basis,source):
    ids=set(ids); mask=temporal.n_id.isin(ids)
    if mask.sum()!=len(ids): raise ValueError("Missing primary IDs")
    for idx in temporal.index[mask]:
        if temporal.at[idx,"chronology_status"]=="undated":
            temporal.at[idx,"composition_min"]=int(lo); temporal.at[idx,"composition_max"]=int(hi)
            temporal.at[idx,"temporal_confidence"]=conf; temporal.at[idx,"temporal_basis"]=basis
            temporal.at[idx,"temporal_source"]=source; temporal.at[idx,"chronology_status"]="primary_dated"
        else:
            old=(int(temporal.at[idx,"composition_min"]),int(temporal.at[idx,"composition_max"]),str(temporal.at[idx,"temporal_confidence"]),str(temporal.at[idx,"temporal_basis"]),str(temporal.at[idx,"temporal_source"]))
            new=(int(lo),int(hi),str(conf),str(basis),str(source))
            if old!=new: raise ValueError(f"Contradictory primary assignment for {temporal.at[idx,'n_id']}: existing={old}, requested={new}")
def assign_sens(ids,lo,hi,basis,source):
    mask=temporal.n_id.isin(set(ids))
    temporal.loc[mask,["sensitivity_min","sensitivity_max"]]=[int(lo),int(hi)]
    temporal.loc[mask,"sensitivity_basis"]=basis; temporal.loc[mask,"sensitivity_source"]=source; temporal.loc[mask,"sensitivity_status"]="sensitivity_only"
def set_circ(ids,year,basis,source):
    for idx in temporal.index[temporal.n_id.isin(set(ids))]:
        cur=temporal.at[idx,"circulation_year"]
        if pd.isna(cur) or int(year)<int(cur):
            temporal.at[idx,"circulation_year"]=int(year); temporal.at[idx,"circulation_basis"]=basis; temporal.at[idx,"circulation_source"]=source
print("Temporal schema initialized:",len(temporal))


In [ ]:
gfile=G/"gongora_obra-poetica.xml"; groot=ET.parse(gfile).getroot()
parent={child:par for par in groot.iter() for child in par}
poems=[]
for el in groot.iter():
    xid=el.attrib.get(XML_ID,"")
    if local(el.tag)=="div" and xid.lower().startswith("poem"):
        ls=[el_text(x) for x in el.iter() if local(x.tag)=="l"]
        ls=[x for x in ls if x]
        if not ls: continue
        vals=[]
        cur=el
        for _ in range(6):
            vals+=list(cur.attrib.values())
            if cur.text: vals.append(cur.text)
            for ch in list(cur):
                if local(ch.tag) in {"head","date","label"}: vals.append(el_text(ch))
                if ch.tail: vals.append(ch.tail)
            cur=parent.get(cur)
            if cur is None: break
        ys=sorted(set(y for v in vals for y in years(v)))
        txt="\n".join(ls)
        poems.append({"g_id":xid,"n_lines":len(ls),"text":txt,"signature":norm(txt),"first_line_sig":norm(ls[0]),
                      "first2_signature":norm("\n".join(ls[:2])),"scholarly_year":ys[0] if len(ys)==1 else pd.NA,
                      "year_status":"unique" if len(ys)==1 else ("ambiguous" if len(ys)>1 else "missing")})
g=pd.DataFrame(poems); g14=g[(g.n_lines==14)&g.signature.ne("")].copy(); g_by_id=g.set_index("g_id",drop=False)
ng=n[n.author_dir.eq("Gongora")].copy()
sig_to_gids=g14.groupby("signature").g_id.apply(list).to_dict()
links=[]
for r in ng.itertuples(index=False):
    ex=sig_to_gids.get(r.signature,[])
    if len(ex)==1: gid,score,method=ex[0],1.0,"exact"
    else:
        best=(None,-1.0)
        for gr in g14.itertuples(index=False):
            sc=SequenceMatcher(None,r.signature,gr.signature).ratio()
            if sc>best[1]: best=(gr.g_id,sc)
        gid,score,method=best[0],best[1],"fuzzy"
    links.append({"n_id":r.n_id,"g_id":gid,"method":method,"score":score,"preaccept":method=="exact" or score>=0.98})
glink=pd.DataFrame(links); pre=glink[glink.preaccept]; collisions=set(pre.g_id.value_counts()[lambda s:s>1].index)
glink["accept_phase4"]=glink.preaccept & ~glink.g_id.isin(collisions)
acc=glink[glink.accept_phase4].merge(g[["g_id","scholarly_year","year_status"]],on="g_id",how="left")
acc=acc[acc.year_status.eq("unique") & acc.scholarly_year.notna()]
for r in acc.itertuples(index=False):
    conf="A" if r.method=="exact" else "B"
    basis="scholarly_chronology_year_exact_link" if r.method=="exact" else "scholarly_chronology_year_fuzzy_link"
    assign_primary([r.n_id],r.scholarly_year,r.scholarly_year,conf,basis,"Cátedra Góngora / Carreira chronology")
print("Góngora phase-4 accepted:",len(acc),"| exact:",int((acc.method=="exact").sum()))


In [ ]:
roman={"I":1,"V":5,"X":10,"L":50,"C":100,"D":500,"M":1000}
def roman_to_int(s):
    total=0; prev=0
    for ch in reversed(str(s).upper()):
        v=roman.get(ch,0); total+=-v if v<prev else v; prev=max(prev,v)
    return total
gar=n[n.author_dir.eq("GarcilasoDeLaVega")].copy()
gar["roman_token"]=gar.title.str.extract(r"^\s*-\s*([IVXLCDM]+)\s*-\s*$",expand=False)
gar["title_no"]=gar.roman_token.map(lambda x:roman_to_int(x) if isinstance(x,str) else pd.NA).astype("Int64")
gar["sonnet_no"]=pd.to_numeric(gar.n_id.str.extract(r"_(\d+)\.xml$",expand=False),errors="coerce").astype("Int64")
assert len(gar)==38 and int((gar.title_no==gar.sonnet_no).fillna(False).sum())==38
GAR={**{i:(1526,1532,"B","scholarly_phase_interval") for i in [1,2,3,4,6,26,27]},
     25:(1534,1535,"B","scholarly_interval"),33:(1535,1535,"A","historically_anchored_scholarly_year"),
     35:(1535,1535,"A","historically_anchored_scholarly_year"),
     **{i:(1533,1535,"B","revised_scholarly_interval") for i in [7,8,12,15,19,28,30,31]}}
gassign=[]
for no,(lo,hi,conf,basis) in GAR.items():
    z=gar[gar.sonnet_no.eq(no)]; assert len(z)==1
    nid=z.iloc[0].n_id
    assign_primary([nid],lo,hi,conf,basis,"Lapesa chronology via Rivers + AISO/AISPI checks")
    gassign.append({"sonnet_no":no,"n_id":nid,"composition_min":lo,"composition_max":hi,"temporal_confidence":conf})
gar_chron=pd.DataFrame(gassign)
print("Garcilaso primary-dated:",len(gar_chron),"/ 38")


In [ ]:
phase4_nids=set(glink.loc[glink.accept_phase4,"n_id"]); phase4_gids=set(glink.loc[glink.accept_phase4,"g_id"])
unmatched=ng[~ng.n_id.isin(phase4_nids)].copy()
first2=g14.groupby("first2_signature").g_id.apply(list).to_dict()
diag=[]
for r in unmatched.itertuples(index=False):
    ids=first2.get(r.first2_signature,[])
    gid=ids[0] if len(ids)==1 else None
    score=pd.NA; yr=pd.NA; status=""
    if gid is not None:
        gr=g_by_id.loc[gid]; score=SequenceMatcher(None,r.signature,gr.signature).ratio(); yr=gr.scholarly_year; status=gr.year_status
    best14=-1.0
    for gr in g14.itertuples(index=False):
        best14=max(best14,SequenceMatcher(None,r.signature,gr.signature).ratio())
    diag.append({"n_id":r.n_id,"first_line":r.first_line,"unique_first2_gid":gid,"unique_first2_score":score,
                 "unique_first2_year":yr,"unique_first2_year_status":status,"best14_score":best14})
gdiag=pd.DataFrame(diag)
gdiag["preaccept"]=gdiag.unique_first2_gid.notna() & pd.to_numeric(gdiag.unique_first2_score,errors="coerce").ge(0.95) & gdiag.unique_first2_year.notna() & gdiag.unique_first2_year_status.eq("unique") & ~gdiag.unique_first2_gid.isin(phase4_gids)
new_collisions=set(gdiag.loc[gdiag.preaccept,"unique_first2_gid"].value_counts()[lambda s:s>1].index)
gdiag["phase5_accept"]=gdiag.preaccept & ~gdiag.unique_first2_gid.isin(new_collisions)
new_g=gdiag[gdiag.phase5_accept].copy()
for r in new_g.itertuples(index=False):
    assign_primary([r.n_id],r.unique_first2_year,r.unique_first2_year,"B","scholarly_chronology_year_variant_link","Cátedra Góngora; unique first-two-line signature + >=0.95 full-text similarity")
gongora_primary=int((temporal.author_dir.eq("Gongora") & temporal.chronology_status.eq("primary_dated")).sum())
print("Phase-5 Góngora recovered:",len(new_g),"| total primary-dated:",gongora_primary)
display(new_g)
assert len(new_g)==5 and gongora_primary==58


In [ ]:
U=HS/"corpus"/"untagged_corpus"
H14=blocks(U/"H.txt"); H14=H14[H14.n_lines.eq(14)]
P214=blocks(U/"P2.txt"); P214=P214[P214.n_lines.eq(14)]
nh=n[n.author_dir.eq("FernandoDeHerrera")].copy()
hm=nh[["n_id","title","first_line","signature"]].copy()
hm["H_exact"]=hm.signature.isin(set(H14.signature)); hm["P2_exact"]=hm.signature.isin(set(P214.signature))
set_circ(hm.loc[hm.H_exact,"n_id"],1582,"H / Algunas obras","Hernández-Lorenzo companion corpus")
set_circ(hm.loc[hm.P2_exact & ~hm.H_exact,"n_id"],1619,"P2 / Versos posthumous","Hernández-Lorenzo companion corpus")
print("Herrera circulation-attested:",int(temporal.loc[temporal.author_dir.eq("FernandoDeHerrera"),"circulation_year"].notna().sum()))
assert int((temporal.author_dir.eq("FernandoDeHerrera") & temporal.chronology_status.eq("primary_dated")).sum())==0

B=blocks(U/"JuanBoscan_Sonetos.txt"); B14=B[B.n_lines.eq(14)]
nb=n[n.author_dir.eq("JuanBoscan")].copy()
boscan_map=nb[["n_id","title","first_line","signature"]].copy(); boscan_map["external_layer_exact"]=boscan_map.signature.isin(set(B14.signature))
assign_sens(nb.n_id,1526,1542,"Boscán Italianate-sonnet activity envelope; sensitivity only","Navagero encounter 1526 to death 1542")
print("Boscán sensitivity-only:",int((temporal.author_dir.eq("JuanBoscan") & temporal.sensitivity_status.eq("sensitivity_only")).sum()))


In [ ]:
gar_worklist=gar[~gar.n_id.isin(set(gar_chron.n_id))][["sonnet_no","n_id","title","first_line","source_bibl"]].sort_values("sonnet_no")
primary=temporal[temporal.chronology_status.eq("primary_dated")].copy()
sens=temporal[temporal.sensitivity_status.eq("sensitivity_only")].copy()
conf=primary.temporal_confidence.value_counts()
assert len(primary)==76
assert int(conf.get("A",0))==14
assert int(conf.get("B",0))==62
assert len(sens)==100
assert int(temporal.circulation_year.notna().sum())==89
assert not primary.temporal_basis.str.contains("publication|witness|edition|circulation",case=False,regex=True).any()
print("Primary-dated:",len(primary),"| A:",int(conf.get("A",0)),"| B:",int(conf.get("B",0)))
print("Sensitivity-only:",len(sens),"| circulation/attestation:",int(temporal.circulation_year.notna().sum()))
print("TEMPORAL INTEGRITY CHECKS: PASSED")
print("CONFIDENCE REGRESSION CHECK: PASSED (A=14, B=62)")

OUT=Path("/content/gasr_phase5_outputs"); OUT.mkdir(exist_ok=True)
temporal.to_csv(OUT/"temporal_master_phase5.csv",index=False)
glink.to_csv(OUT/"gongora_phase4_links.csv",index=False)
gdiag.to_csv(OUT/"gongora_phase5_recovery_diagnostics.csv",index=False)
gar_chron.to_csv(OUT/"garcilaso_chronology_seed.csv",index=False)
gar_worklist.to_csv(OUT/"garcilaso_unassigned_worklist.csv",index=False)
hm.to_csv(OUT/"herrera_textual_layer_map.csv",index=False)
boscan_map.to_csv(OUT/"boscan_layer_audit.csv",index=False)
print()
print("PHASE 5 CORRECTED CHECKPOINT")
print("----------------------------")
print("Expected verified totals: primary=76; A=14; B=62; sensitivity=100; circulation=89.")
print("Save this executed notebook to GitHub.")
print("Do NOT build semantic networks yet.")
